In [1]:
from scipy.io import loadmat
import numpy as np
from scipy.stats import zscore
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr

In [3]:
from sklearn.metrics import explained_variance_score
from sklearn.cross_decomposition import PLSRegression
from numpy.random import randint

# An old config used to check Nguyen's data
dirIn = '/home/niklasz/Desktop/ucla/research_work/nguyen_data_analysis/2023_06_22/Data'
checkpoint_range_dir = '/home/niklasz/Desktop/ucla/research_work/nguyen_data_analysis/2023_06_22/Data'
example_file = 'behavior_output_100iters_test_1_v2.mat'
example_path = f'{dirIn}/{example_file}'
data_mat = loadmat(example_path)
episode_length, activations_count = data_mat['R1'].shape
plsc_levels = 3  # whether to only get PLSC1, PLSC2, PLSC3, etc.
episode_count = 5
n_temporal_shuffles = 100
checkpoint_iters = [100, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
train_iterations = len(checkpoint_iters)
shifted_indices = np.roll(np.arange(0, episode_count), 10)
n_cca_components = 6 # this is just picked to be the same as Matlab
null_dist_shuffles = 100
shuffle_distance = 10
pct_cutoff = 97.5

# X1 = np.zeros((train_iterations, episode_count, episode_length, activations_count), dtype=float)
# X2 = np.zeros((train_iterations, episode_count, episode_length, activations_count), dtype=float)
activation_masks = np.ones((train_iterations, episode_count, episode_length), dtype=bool)
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for train_idx in range(0, train_iterations):
    for ep_idx in range(0, episode_count):
        file_name = f'behavior_output_{checkpoint_iters[train_idx]}iters_test_{ep_idx+1}_v2.mat'
        file_path = f'{dirIn}/{file_name}'
        data_mat = loadmat(file_path)

        # Behavioural arrays
        p1_action_bits = np.equal.outer(data_mat['action1'].squeeze(), np.array([0, 1, 2, 3])).astype(np.float64)
        p1_behaviours_1 =  data_mat['Bv1'][:, [1,3,4,5,6,7]].astype(np.float64)
        p1_position = data_mat['Pos_R1']
        Bv_1 = np.hstack([p1_action_bits, p1_behaviours_1, p1_position])
        Bv_1_z = np.nan_to_num(zscore(Bv_1, ddof=1))

        p2_action_bits = np.equal.outer(data_mat['action2'].squeeze(), np.array([0, 1, 2, 3])).astype(np.float64)
        p2_behaviours_1 =  data_mat['Bv2'][:, [1,3,4,5,6,7]].astype(np.float64)
        p2_position = data_mat['Pos_R2']
        Bv_2 = np.hstack([p2_action_bits, p2_behaviours_1, p2_position])
        Bv_2_z = np.nan_to_num(zscore(Bv_2, ddof=1))
        # TODO is num components related to matrix rank?
        cca = CCA(n_components=n_cca_components)
        # Get canonical values
        bv1_values, bv2_values = cca.fit_transform(X=Bv_1_z, y=Bv_2_z)
        # Yields a correlation for every possible bv1 and bv2 component pair. Sorted by descending
        # correlation.
        pearson_correlations = [pearsonr(bv1_values[:,c],bv2_values[:,c])[0] for c in range(n_cca_components)]

        # Permute behaviours and run it again
        permuted_correlations = []
        for i in range(null_dist_shuffles):
            rand_low = shuffle_distance
            rand_high = Bv_1_z.shape[0]
            bv1_p_values, bv2_p_values = cca.fit_transform(X=np.roll(Bv_1_z,randint(rand_low, rand_high)),
                                                       y=np.roll(Bv_2_z,randint(rand_low, rand_high)))
            permuted_correlations += [pearsonr(bv1_p_values[:,c],bv2_p_values[:,c])[0] for c in range(n_cca_components)]

        cut_off_val = np.percentile(permuted_correlations, pct_cutoff)
        significant_dimensions = np.sum(pearson_correlations > cut_off_val)
        # If there are any correlations greater than random permutation use PLSR
        if significant_dimensions > 0:
            def get_plsr_for(X: np.ndarray, y: np.ndarray) -> float:
                pls = PLSRegression(n_components=significant_dimensions, max_iter=1000)
                pls.fit(X, y)
                pred_y = pls.predict(X)
                return explained_variance_score(y, pred_y)
            # TODO let Y be activations in our case
            rand_low = shuffle_distance
            rand_high = Bv_1_z.shape[0]
            # TODO store VarExp somewhere
            get_plsr_for(X=bv1_values, y=np.roll(Bv_1_z,randint(rand_low, rand_high)))


/home/niklasz/Desktop/ucla/research_work/social-agents/venv/lib/python3.9/site-packages/sklearn/cross_decomposition/_pls.py:300: UserWarning: Y residual is constant at iteration 5
  warnings.warn(f"Y residual is constant at iteration {k}")
/home/niklasz/Desktop/ucla/research_work/social-agents/venv/lib/python3.9/site-packages/scipy/stats/_stats_py.py:4068: PearsonRConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(PearsonRConstantInputWarning())
/home/niklasz/Desktop/ucla/research_work/social-agents/venv/lib/python3.9/site-packages/sklearn/cross_decomposition/_pls.py:300: UserWarning: Y residual is constant at iteration 5
  warnings.warn(f"Y residual is constant at iteration {k}")
/home/niklasz/Desktop/ucla/research_work/social-agents/venv/lib/python3.9/site-packages/scipy/stats/_stats_py.py:4068: PearsonRConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(PearsonRConsta